# Ejercicio 4
Regresión logística y disponibilidad temporal de las variables.

In [8]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
from lista4.data import load_csv
from lista4.models import fit_logistic, logistic_probabilities, logistic_classes
from lista4.metrics import logistic_risk, classification_errors, classification_error_rate

df = load_csv("clientes.csv")
df.head()


,antiguedad_meses,uso_30d_previos,facturas_vencidas_180d,tickets_90d,pago_automatico,ajuste_comercial_30d,abandona_30d
0,66,37.927153,0,1,0,23.039078,0
1,12,46.656223,0,1,1,19.158849,0
2,49,22.990662,0,3,1,27.712278,0
3,49,24.546478,1,0,0,96.997980,1
4,30,64.413222,0,2,1,0.000000,0


In [9]:
variables_disponibles = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico"
]

todas_las_predictoras = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico",
    "ajuste_comercial_30d"
]

target = "abandona_30d"

variables_disponibles


['antiguedad_meses',
 'uso_30d_previos',
 'facturas_vencidas_180d',
 'tickets_90d',
 'pago_automatico']

Las variables que ya se conocen en t0 son: `antiguedad_meses`, `uso_30d_previos`,
`facturas_vencidas_180d`, `tickets_90d` y `pago_automatico`.

Dejamos por fuera `ajuste_comercial_30d` porque esa variable se construye con
datos entre t0 y t0+30, o sea que todavía no existe cuando el modelo tiene que
predecir.

In [10]:
M1 = fit_logistic(df, todas_las_predictoras, target)
M2 = fit_logistic(df, variables_disponibles, target)

p1 = logistic_probabilities(M1, df, todas_las_predictoras)
p2 = logistic_probabilities(M2, df, variables_disponibles)

y1 = (p1 >= 0.5).astype(int)
y2 = (p2 >= 0.5).astype(int)

resultados = pd.DataFrame({
    "modelo": ["M1 - todas", "M2 - disponibles"],
    "riesgo_logistico": [
        logistic_risk(df[target], p1),
        logistic_risk(df[target], p2)
    ],
    "errores_clasificacion": [
        classification_errors(df[target], y1),
        classification_errors(df[target], y2)
    ],
    "proporcion_errores": [
        classification_error_rate(df[target], y1),
        classification_error_rate(df[target], y2)
    ]
})

resultados


,modelo,riesgo_logistico,errores_clasificacion,proporcion_errores
0,M1 - todas,0.342485,160,0.133333
1,M2 - disponibles,0.571716,336,0.280000


M1 usa todas las variables: `antiguedad_meses`, `uso_30d_previos`,
  `facturas_vencidas_180d`, `tickets_90d`, `pago_automatico` y
  `ajuste_comercial_30d`.
M2 usa solo las disponibles en t0: las mismas menos `ajuste_comercial_30d`.

M1: riesgo logístico 0.342, 160 errores, tasa de error 0.133.
M2: riesgo logístico 0.572, 336 errores, tasa de error 0.280.

M1 tiene mejores métricas que M2 en todo. Pero hay que tener cuidado: esa mejora
viene de usar `ajuste_comercial_30d`, que no está disponible en t0.

La tabla reúne los resultados: riesgo logístico, número de errores y proporción
de errores de cada modelo. M1 le gana a M2 en las tres columnas.

### Respuesta (e)
Si el modelo debe ejecutarse exactamente en t0, el único que podemos usar es M2,
porque solo usa variables que ya existen en ese momento.

M1 no se puede usar porque incluye `ajuste_comercial_30d`, que se construye con
datos entre t0 y t0+30. En el momento de hacer la predicción esa variable todavía
no existe, entonces usarla sería hacer trampa. El modelo se ve mejor en el papel
(menor riesgo y menos errores), pero en producción no se podría calcular.

La variable que produce la diferencia entre M1 y M2 es `ajuste_comercial_30d`.
Es la única que está en M1 y no en M2, y es la razón por la que M1 parece mejor.